In [1]:
import pandas as pd
import glob
import os

print("--- Step 1: Loading and Stitching the Raw BoT-IoT Dataset ---")

# Point this to the folder containing all 4 CSVs
data_dir = "../data/raw/bot_iot/" 

# Use glob to automatically find all 4 files that match the pattern
all_files = glob.glob(os.path.join(data_dir, "UNSW_2018_IoT_Botnet_Full5pc_*.csv"))

print(f"Found {len(all_files)} BoT-IoT chunks. Stitching them together...")

# Loop through, read each one, and append it to a list
df_list = []
for file in all_files:
    print(f"Loading {os.path.basename(file)}...")
    df_chunk = pd.read_csv(file)
    df_list.append(df_chunk)

# Concatenate all chunks into one massive, unified matrix
df_bot = pd.concat(df_list, ignore_index=True)

print("\nSuccess! All chunks unified.")
print(f"Total Network Packets Loaded: {len(df_bot):,}")

--- Step 1: Loading and Stitching the Raw BoT-IoT Dataset ---
Found 4 BoT-IoT chunks. Stitching them together...
Loading UNSW_2018_IoT_Botnet_Full5pc_1.csv...


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_17104\2690437591.py:19: DtypeWarning: Columns (0: sport, 1: dport) have mixed types. Specify dtype option on import or set low_memory=False.
  df_chunk = pd.read_csv(file)


Loading UNSW_2018_IoT_Botnet_Full5pc_2.csv...
Loading UNSW_2018_IoT_Botnet_Full5pc_3.csv...
Loading UNSW_2018_IoT_Botnet_Full5pc_4.csv...

Success! All chunks unified.
Total Network Packets Loaded: 3,668,522


In [2]:
import numpy as np

print("--- Step 2: The Universal Translation Engine ---")

# 1. The Rosetta Stone: Dictionary mapping BoT-IoT names to ToN-IoT standard names
schema_mapping = {
    'sbytes': 'src_bytes',
    'dbytes': 'dst_bytes',
    'spkts': 'src_pkts',
    'dpkts': 'dst_pkts',
    'dur': 'duration',
    'TnBPSrcIP': 'src_ip_bytes',
    'TnBPDstIP': 'dst_ip_bytes',
    'state': 'conn_state',
    'proto': 'proto',
    'attack': 'label' # Standardizing the target variable
}

# Apply the translation
df_bot_translated = df_bot.rename(columns=schema_mapping)

# 2. Injecting the Missing "Ghost" Columns
print("Injecting missing ToN-IoT features (DNS metrics) as zeros...")
missing_columns = ['dns_qclass', 'dns_qtype', 'dns_rcode']

for col in missing_columns:
    df_bot_translated[col] = 0

# 3. Filter down to ONLY the baseline columns our AI knows how to read
# We drop things like 'pkSeqID' or 'stime' because the AI was never trained on them.
final_columns = [
    'src_bytes', 'dst_bytes', 'src_pkts', 'dst_pkts', 'duration',
    'src_ip_bytes', 'dst_ip_bytes', 'conn_state', 'proto',
    'dns_qclass', 'dns_qtype', 'dns_rcode', 'label'
]

# Create the final mapped dataframe
df_bot_final = df_bot_translated[final_columns]

print("\n--- Translation Complete! ---")
print("New Standardized Schema:")
print(df_bot_final.dtypes)

--- Step 2: The Universal Translation Engine ---
Injecting missing ToN-IoT features (DNS metrics) as zeros...

--- Translation Complete! ---
New Standardized Schema:
src_bytes         int64
dst_bytes         int64
src_pkts          int64
dst_pkts          int64
duration        float64
src_ip_bytes      int64
dst_ip_bytes      int64
conn_state          str
proto               str
dns_qclass        int64
dns_qtype         int64
dns_rcode         int64
label             int64
dtype: object


In [3]:
print("--- Step 3: Saving the Universal Schema ---")

# Define the output path
output_file = os.path.join(data_dir, "bot_iot_mapped.csv")

# Save the dataset
print(f"Saving translated dataset to: {output_file}...")
df_bot_final.to_csv(output_file, index=False)

print("\nSuccess! BoT-IoT is now officially wearing the ToN-IoT Mask.")
print("It is ready for the Assembly Line.")

--- Step 3: Saving the Universal Schema ---
Saving translated dataset to: ../data/raw/bot_iot/bot_iot_mapped.csv...

Success! BoT-IoT is now officially wearing the ToN-IoT Mask.
It is ready for the Assembly Line.
